# G5: first GPU test of gemma-4-12b-it-qat-w4a16-ct (base model, no adapter): serving, decode speed, 4-task smoke, one 75-task round

Compare with the 31B rounds r1/r2 (25/75, 28/75 resolved; 44/39 capped). Output: `g5/g5_summary.json`, `g5/capture.jsonl.gz`, `g5/b12/`.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

## Setup

In [ ]:
import json, os, random, shutil, sys, threading, time, traceback
from pathlib import Path
WORKING_DIR = Path('/kaggle/working'); WORKING_DIR.mkdir(parents=True, exist_ok=True)
G2 = WORKING_DIR / 'g5'; G2.mkdir(exist_ok=True); CODE = G2 / 'code'; CODE.mkdir(exist_ok=True)
RESULTS = {'started': time.strftime('%Y-%m-%d %H:%M:%S')}
def save_results(): (G2 / 'g5_summary.json').write_text(json.dumps(RESULTS, indent=1, default=str))
BUDGET_MIN = 160; T_START = time.time()
def _watchdog():   # a hung round must not burn GPU quota
    RESULTS['watchdog'] = f'fired after {BUDGET_MIN} min'; save_results(); os._exit(3)
_wd = threading.Timer(BUDGET_MIN * 60, _watchdog); _wd.daemon = True; _wd.start()
for n, txt in {"capture_proxy.py": "\"\"\"Transparent logging reverse proxy between the harness (LiteLLM -> ADK) and vLLM, for G2 data collection.\n\nFor every non-streaming POST .../chat/completions it records, gzip-compressed, one JSON line with the FULL request body (messages incl. reasoning,\ntools, chat_template_kwargs, sampling params) and the FULL response body. It asks vLLM for `return_token_ids` so the record also carries the exact\nprompt token ids and completion token ids (no re-rendering guesses); if the server rejects that parameter it retries without it and keeps going.\nThe harness receives the response with the token-id fields stripped, i.e. exactly what it would get without the proxy.\nEverything else (GET /v1/models, /health, streaming requests) is passed through untouched.\n\"\"\"\n\nimport gzip\nimport hashlib\nimport json\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nimport requests\n\n\ndef task_key(messages: list[dict]) -> str:\n    \"\"\"Stable id of the agent session a request belongs to: hash of the system prompt + the first user message (the task statement).\"\"\"\n    sys_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"system\"), \"\") or \"\"\n    usr_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"user\"), \"\") or \"\"\n    h = hashlib.sha1()\n    for part in (sys_txt, usr_txt):\n        h.update((part if isinstance(part, str) else json.dumps(part, sort_keys=True)).encode(\"utf-8\", \"ignore\"))\n        h.update(b\"\\x00\")\n    return h.hexdigest()[:16]\n\n\nclass CaptureProxy:\n    def __init__(self, upstream: str, out_path: str, port: int = 8001, host: str = \"127.0.0.1\", ask_token_ids: bool = True):\n        self.upstream = upstream.rstrip(\"/\")          # e.g. http://127.0.0.1:8000\n        self.out_path = out_path\n        self.port, self.host = port, host\n        self.ask_token_ids = ask_token_ids\n        self.round_tag = \"r0\"\n        self.tokenize_fallback = False        # if True and vLLM returns no token ids, ask /tokenize for the exact prompt ids instead\n        self.n_records = 0\n        self.n_with_ids = 0\n        self.n_errors = 0\n        self._lock = threading.Lock()\n        self._httpd: ThreadingHTTPServer | None = None\n\n    # ---------------------------------------------------------------- lifecycle\n    def start(self) -> \"CaptureProxy\":\n        proxy = self\n\n        class H(BaseHTTPRequestHandler):\n            protocol_version = \"HTTP/1.1\"\n\n            def log_message(self, *a):  # silence\n                pass\n\n            def _relay(self, status: int, body: bytes, ctype: str = \"application/json\"):\n                self.send_response(status)\n                self.send_header(\"Content-Type\", ctype)\n                self.send_header(\"Content-Length\", str(len(body)))\n                self.end_headers()\n                self.wfile.write(body)\n\n            def do_GET(self):\n                try:\n                    r = requests.get(proxy.upstream + self.path, timeout=60)\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                except Exception as e:  # noqa: BLE001\n                    self._relay(502, json.dumps({\"error\": str(e)}).encode())\n\n            def do_POST(self):\n                raw = self.rfile.read(int(self.headers.get(\"Content-Length\", \"0\")))\n                try:\n                    req = json.loads(raw) if raw else {}\n                except Exception:  # noqa: BLE001\n                    req = None\n                is_chat = self.path.rstrip(\"/\").endswith(\"/chat/completions\") and isinstance(req, dict) and not req.get(\"stream\")\n                if not is_chat:                      # pass-through (streaming, other endpoints)\n                    try:\n                        r = requests.post(proxy.upstream + self.path, data=raw, headers={\"Content-Type\": \"application/json\"}, timeout=1800)\n                        self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                    except Exception as e:  # noqa: BLE001\n                        self._relay(502, json.dumps({\"error\": str(e)}).encode())\n                    return\n                t0 = time.time()\n                asked = proxy.ask_token_ids           # decided ONCE per request (another thread may flip the flag meanwhile)\n                body = dict(req, return_token_ids=True) if asked else req\n                try:\n                    r = requests.post(proxy.upstream + self.path, json=body, timeout=1800)\n                    if asked and r.status_code == 400 and \"return_token_ids\" in r.text:\n                        proxy.ask_token_ids = False   # this vLLM rejects the parameter: stop asking, redo this call plainly\n                        r = requests.post(proxy.upstream + self.path, json=req, timeout=1800)\n                except Exception as e:  # noqa: BLE001\n                    proxy.n_errors += 1\n                    self._relay(502, json.dumps({\"error\": f\"proxy upstream failure: {e}\"}).encode())\n                    return\n                try:\n                    resp = r.json()\n                except Exception:  # noqa: BLE001\n                    resp = None\n                if r.status_code == 200 and isinstance(resp, dict):\n                    proxy._record(req, resp, t0, time.time())\n                    clean = {k: v for k, v in resp.items() if k != \"prompt_token_ids\"}\n                    clean[\"choices\"] = [{k: v for k, v in ch.items() if k != \"token_ids\"} for ch in resp.get(\"choices\", [])]\n                    self._relay(200, json.dumps(clean).encode())\n                else:\n                    proxy.n_errors += 1\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n\n        self._httpd = ThreadingHTTPServer((self.host, self.port), H)\n        self._httpd.daemon_threads = True\n        threading.Thread(target=self._httpd.serve_forever, daemon=True).start()\n        return self\n\n    def stop(self) -> None:\n        if self._httpd:\n            self._httpd.shutdown()\n            self._httpd.server_close()\n\n    # ---------------------------------------------------------------- recording\n    def _record(self, req: dict, resp: dict, t0: float, t1: float) -> None:\n        has_ids = bool(resp.get(\"prompt_token_ids\")) and bool((resp.get(\"choices\") or [{}])[0].get(\"token_ids\"))\n        rec = {\"round\": self.round_tag, \"task_key\": task_key(req.get(\"messages\", [])), \"t0\": round(t0, 3), \"t1\": round(t1, 3), \"request\": req, \"response\": resp}\n        if not has_ids and self.tokenize_fallback:\n            try:\n                tk = {k: req[k] for k in (\"model\", \"messages\", \"tools\", \"chat_template_kwargs\") if k in req}\n                tk[\"add_generation_prompt\"] = True\n                r = requests.post(self.upstream + \"/tokenize\", json=tk, timeout=300)\n                if r.status_code == 200:\n                    rec[\"prompt_token_ids_tokenize\"] = r.json().get(\"tokens\")\n            except Exception:  # noqa: BLE001\n                pass\n        line = (json.dumps(rec, ensure_ascii=False) + \"\\n\").encode(\"utf-8\")\n        with self._lock:                             # one gzip member per record; the concatenation is a valid gzip stream\n            with open(self.out_path, \"ab\") as f:\n                f.write(gzip.compress(line, compresslevel=3))\n            self.n_records += 1\n            self.n_with_ids += has_ids\n\n    def stats(self) -> dict:\n        return {\"records\": self.n_records, \"with_token_ids\": self.n_with_ids, \"errors\": self.n_errors, \"asking_token_ids\": self.ask_token_ids, \"round\": self.round_tag}\n\n\ndef read_records(path: str):\n    \"\"\"Iterate the records of a capture file.\"\"\"\n    with gzip.open(path, \"rt\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                yield json.loads(line)\n"}.items(): (CODE / n).write_text(txt, encoding='utf-8')
sys.path.insert(0, str(CODE))
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
arm_dirs = [p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_scaffold_v2_b1' in p.read_text(encoding='utf-8')]
assert len(arm_dirs) == 1, f'expected exactly one B1 arm, found {arm_dirs}'
ARM = WORKING_DIR / 'arm'; shutil.rmtree(ARM, ignore_errors=True); shutil.copytree(arm_dirs[0], ARM)
for _rel in ('agent.yaml', 'sub_agents/code_analyzer.yaml'):   # one variant for every agent and sub-agent
    _p = ARM / _rel; _t = _p.read_text(encoding='utf-8'); assert _t.count('gemma-4-31b-it-qat-w4a16-ct') == 1, _rel; _p.write_text(_t.replace('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-12b-it-qat-w4a16-ct'), encoding='utf-8', newline='\n')
assert 'gemma-4-31b-it-qat-w4a16-ct' not in (ARM / 'agent.yaml').read_text()
for must in ('agent.yaml', 'prompts/system.md', 'prompts/analyzer.md', 'configs/sampling.yaml', 'sub_agents/code_analyzer.yaml', 'eval_config.yaml'):
    assert (ARM / must).is_file(), f'arm missing {must}'
print('arm:', arm_dirs[0], '|', (ARM / 'configs/sampling.yaml').read_text().replace(chr(10), ' ')[-120:])
from swegemma.models import load_tasks
ALL_IDS = ["fastapi_11194", "fastapi_13537", "fastapi_13920", "fastapi_14077", "fastapi_14258", "fastapi_14301", "fastapi_14306", "fastapi_14372", "fastapi_14419", "fastapi_14448", "fastapi_14458", "fastapi_14463", "fastapi_14479", "fastapi_14487", "fastapi_14492", "fastapi_14616", "fastapi_14786", "fastapi_14794", "fastapi_14851", "fastapi_14873", "fastapi_14986", "fastapi_15280", "fastapi_15588", "fastapi_15589", "fastapi_15661", "fastapi_15800", "fastapi_5077", "fastapi_5624", "fastapi_9425", "fastapi_9555", "fastapi_9753", "requests_6644", "rich_2725", "rich_2943", "rich_3006", "rich_3043", "rich_3052", "rich_3061", "rich_3063", "rich_3067", "rich_3105", "rich_3180", "rich_3278", "rich_3454", "rich_3468", "rich_3470", "rich_3471", "rich_3472", "rich_3480", "rich_3506", "rich_3518", "rich_3521", "rich_3535", "rich_3675", "rich_3676", "rich_3718", "rich_3772", "rich_3777", "rich_3782", "rich_3882", "rich_3894", "rich_3905", "rich_3930", "rich_3934", "rich_3935", "rich_3938", "rich_3942", "rich_3944", "rich_3953", "rich_4006", "rich_4070", "rich_4075", "rich_4076", "rich_4077", "rich_4079"]
by_id = {t.instance_id: t for t in load_tasks(TASKS_PATH)}
missing = [i for i in ALL_IDS if i not in by_id]
assert not missing, f'task ids missing from the dataset: {missing}'
SMOKE_IDS = random.Random(5).sample([i for i in ALL_IDS if i.startswith(('fastapi_', 'rich_'))], 4)
print(len(ALL_IDS), 'tasks;', 'smoke ids:', SMOKE_IDS)

## vLLM (real 31B, no adapter) + capture proxy

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-12b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-12b-it-qat-w4a16-ct/2')
assert MODEL_PATH.is_dir(), MODEL_PATH
first_arm = ARM
try:   # swegemma >= 0.2.11 / adk_submission >= 0.2.13 (wheelhouse of 2026-10-09)
    from adk_submission import discover_declared_models
    _declared = discover_declared_models(first_arm); assert len(_declared) == 1, f'expected exactly one declared model, got {_declared}'
    declared_model = next(iter(_declared))
except ImportError:   # older wheelhouse
    from swegemma.models.discovery import validate_single_declared_model
    declared_model = validate_single_declared_model(first_arm)
adapters = discover_adapters(str(first_arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=False,
    extra_args=('--hf-overrides', json.dumps({'architectures': ['Gemma4ForCausalLM'], 'model_type': 'gemma4'})),
    tensor_parallel_size=tp_size, startup_timeout=60 * 12)
# --- guarded patch: Gemma4ForCausalLM.load_weights must skip vision tensors (12B checkpoint carries `vision_embedder.*`; see forum #748287)
import re as _re, vllm as _vllm
_g4 = Path(_vllm.__file__).parent / 'model_executor' / 'models' / 'gemma4.py'
_src = _g4.read_text(encoding='utf-8')
print('gemma4.py: mentions vision_embedder:', 'vision_embedder' in _src, '| lines', len(_src.splitlines()))
_marker = '                name = name.replace("language_model.", "")\n'
if 'vision_embedder' not in _src and _src.count(_marker) == 1:
    _src = _src.replace(_marker, _marker + '                if name.startswith(("model.vision_embedder.", "vision_embedder.", "embed_vision.", "model.embed_vision.")):\n                    continue   # patched by G5: text-only model\n')
    _g4.write_text(_src, encoding='utf-8'); print('patched', _g4)
else:
    print('NOT patching (already handles vision_embedder, or the insertion point is not unique):', _src.count(_marker))
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
import capture_proxy as CP
VLLM_URL = f'http://127.0.0.1:{server_instance.config.port}'
proxy = CP.CaptureProxy(VLLM_URL, str(G2 / 'capture.jsonl.gz'), port=8001).start()
server_instance.config.port = 8001      # the model registry reads base_url at creation time: from now on the harness talks to the capture proxy
models = server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')
print('harness endpoint (proxy):', server_instance.base_url, '-> upstream', VLLM_URL)

# --- single-stream decode speed (31B reference with no adapter: 37.5 tok/s at concurrency 1)
import requests as _rq
def _probe(n_tok=400):
    best = 0
    for _ in range(3):
        t0 = time.perf_counter()
        # vLLM serves the model under its PATH (captured 31B requests carry the path as `model`), not under the alias
        r = _rq.post(VLLM_URL + '/v1/chat/completions', json={'model': str(MODEL_PATH), 'max_tokens': n_tok, 'temperature': 0.7,
              'messages': [{'role': 'user', 'content': 'Write a long, detailed explanation of how Python decorators work, with several examples.'}],
              'chat_template_kwargs': {'enable_thinking': False}}, timeout=600).json()
        dt = time.perf_counter() - t0
        if 'usage' not in r: raise RuntimeError('probe response without usage: ' + json.dumps(r)[:400])
        best = max(best, r['usage']['completion_tokens'] / dt)
    return round(best, 1)
try:   # informational only: must never stop the run
    RESULTS['decode_tok_s_single_stream'] = _probe(); print('decode tok/s (single stream):', RESULTS['decode_tok_s_single_stream'], '| 31B reference 37.5')
except Exception as _e:
    RESULTS['decode_probe_error'] = repr(_e)[:600]; print('decode probe FAILED (non-fatal):', repr(_e)[:600])
save_results()


## Smoke round, capture gate, full rounds

In [ ]:
import asyncio, concurrent.futures, requests, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_fn):
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())

def budget_of(arm_dir):
    cfg = yaml.safe_load((arm_dir / 'eval_config.yaml').read_text(encoding='utf-8')).get('evaluation', {})
    out = {}
    for k, cast in (('timeout_seconds', int), ('max_tool_calls', int), ('max_time_minutes', float)):
        if k in cfg: out[k] = cast(cfg[k])
    t = cfg.get('max_turns', cfg.get('max_llm_calls'))
    if t is not None: out['max_turns'] = int(t)
    return out

limits, gen_constraints = build_submission_limits()
print('per-task budget:', budget_of(ARM))

def run_round(tag, ids):
    proxy.round_tag = tag
    rd = G2 / tag
    cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=rd, submission_dir=ARM, models=models, sandbox='subprocess',
        task_ids=list(ids), concurrency=4, display_mode='quiet', limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'), wheels_dir=DATA_DIR / 'wheels', verbose=False, **budget_of(ARM))
    t0 = time.perf_counter()
    try:
        run_sync(Evaluator(cfg).run)
    except Exception:
        traceback.print_exc(); RESULTS[tag] = {'error': 'run raised', 'trace': traceback.format_exc()[-1500:]}; save_results(); return None
    rows = [json.loads(l) for l in open(rd / 'task_results.jsonl', encoding='utf-8')]
    RESULTS[tag] = {'n': len(rows), 'resolved': sum(r['resolved'] for r in rows), 'patch_written': sum(1 for r in rows if r['agent_patch_size'] > 0),
                    'time_cap_hit': sum(1 for r in rows if 'session timeout' in (r['error'] or '')), 'wall_s': round(time.perf_counter() - t0, 1),
                    'mean_task_s': round(sum(r['duration_seconds'] for r in rows) / max(1, len(rows)), 1), 'resolved_ids': sorted(r['instance_id'] for r in rows if r['resolved']),
                    'proxy': proxy.stats()}
    save_results(); print(tag, {k: v for k, v in RESULTS[tag].items() if k != 'resolved_ids'}, flush=True)
    return rows

# ------------------------------------------------------------------ SMOKE: 4 tasks, then a fail-fast capture gate (saves quota if capture is broken)
rows = run_round('smoke', SMOKE_IDS)
st = proxy.stats(); print('capture after smoke:', st)
gate = {'rows': bool(rows) and len(rows) == len(SMOKE_IDS), 'records >= 12': st['records'] >= 12, 'no proxy errors': st['errors'] == 0}
if st['with_token_ids'] == 0 and st['records'] > 0:      # vLLM returned no ids: switch on the /tokenize fallback for the rest of the run
    proxy.tokenize_fallback = True; print('WARNING: no token ids from vLLM; /tokenize fallback enabled')
try:   # cross-check: ids returned by vLLM == ids from /tokenize for the same request (the exactness of the whole capture rests on this)
    first = next(CP.read_records(str(G2 / 'capture.jsonl.gz')))
    rq = first['request']
    tk = {k: rq[k] for k in ('model', 'messages', 'tools', 'chat_template_kwargs') if k in rq}; tk['add_generation_prompt'] = True
    tok = requests.post(VLLM_URL + '/tokenize', json=tk, timeout=300).json().get('tokens')
    got = first['response'].get('prompt_token_ids') or first.get('prompt_token_ids_tokenize')
    RESULTS['token_id_crosscheck'] = {'tokenize_len': len(tok or []), 'captured_len': len(got or []), 'identical': tok == got}
except Exception:
    RESULTS['token_id_crosscheck'] = {'error': traceback.format_exc()[-800:]}
print('cross-check:', RESULTS['token_id_crosscheck']); save_results()
RESULTS['smoke_gate'] = gate
if not all(gate.values()):
    raise RuntimeError(f'SMOKE GATE FAILED, stopping before the expensive rounds: {gate} {st}')
print('smoke gate passed:', gate)

# ------------------------------------------------------------------ one full round over all 75 tasks with the 12B (31B reference r1/r2: 25/75 and 28/75)
ids = list(ALL_IDS); random.Random(101).shuffle(ids)
run_round('b12', ids)

print('\n===== G5 (12B) SUMMARY =====')
for tag in ('smoke', 'b12'):
    if tag in RESULTS and 'n' in RESULTS[tag]: print(tag, {k: v for k, v in RESULTS[tag].items() if k not in ('resolved_ids', 'proxy')})
print('capture:', proxy.stats(), '| file MB:', round((G2 / 'capture.jsonl.gz').stat().st_size / 1e6, 1))
RESULTS['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); RESULTS['capture'] = proxy.stats(); save_results()
_wd.cancel()